# MultiVI trimodal integration — Patient 1 (03H096 / PB2)

**Role in the paper:** Core Methods step for Figure 1 (joint RNA + ATAC + protein embedding).

**What this notebook does**
1. Loads the raw multimodal MuData for Patient 1
2. Applies RNA / ATAC / protein QC filters (including removal of IgG isotype-control ADTs)
3. Trains **MultiVI** on the paired RNA + ATAC matrix with protein as a side channel
4. Builds neighbours and a starting Leiden clustering (resolution 0.8)
5. Inspects chromatin accessibility per cluster — this is how the high-accessibility root / CSC cluster is found
6. Runs **CellTypist** as the first of several complementary checks for non-leukemic cells
7. **Writes the cleaned MultiVI AnnData** used by downstream analyses

**This workflow requires a sequence of analytical steps rather than a single automated operation. For the Patient 1 example, Leiden clustering is performed at resolution 0.8. The high-accessibility cluster is identified by comparing ATAC-seq counts per cell across clusters, and its root-like position is subsequently assessed by trajectory analysis.**

**Objects**
- **Reads:** `DATA_DIR / "Teaseq_PB2.h5mu"` (raw trimodal MuData)
- **Creates:** `DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/MultiVI_Patient1_Relapse_03h096_Trimodal_Cleaned.h5ad"`, also distributed as `Teaseq_Multi_VI_PB2_Cleaned.h5ad`, plus the trained MultiVI model directory

The same notebook was run per patient; only the input MuData and the output file name change.


## Paths and settings

In [ ]:
from pathlib import Path

# Root of the companion data package. Point this at your local copy.
DATA_DIR = Path("PATH_TO_DATA")  # <-- set this to your local data root
OUT_DIR = DATA_DIR / "outputs/MultiVI_PB2"     # figures and tables written by this notebook
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os

import anndata
import anndata as ad
import matplotlib.pyplot as plt
import muon
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns

## Load the raw trimodal MuData

In [ ]:
teapb2 = muon.read(DATA_DIR / "Teaseq_PB2.h5mu")
teapb2.var_names_make_unique()

In [ ]:
# Update indices for teapb2
teapb2.obs.index = [name + '_PB2' for name in teapb2.obs_names]
teapb2.mod['rna'].obs.index = [name + '_PB2' for name in teapb2.mod['rna'].obs_names]
teapb2.mod['atac'].obs.index = [name + '_PB2' for name in teapb2.mod['atac'].obs_names]
teapb2.mod['protein'].obs.index = [name + '_PB2' for name in teapb2.mod['protein'].obs_names]

## Restrict to the QC-passing barcodes

Doublets and low-quality cells were removed in a first pass. The barcode list is stored in the cleaned MultiVI object written at the end of this notebook, and is reused here so that the published embedding is reproducible from the raw MuData.

On a **first run** (no cleaned object yet), skip this cell and apply the RNA / ATAC / protein filters below instead.


In [ ]:
import muon
import anndata as ad

# Load the cleaned objects
cleaned_teapb2 = ad.read_h5ad(DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/MultiVI_Patient1_Relapse_03h096_Trimodal_Cleaned.h5ad")

# Extract the cell indices from the cleaned objects
cleaned_teapb2_cell_ids = cleaned_teapb2.obs.index.tolist()

# Filter the original objects to keep only the cells present in their cleaned counterparts
teapb2 = teapb2[teapb2.obs.index.isin(cleaned_teapb2_cell_ids)]

# Print the dimensions of the filtered objects
print(f"Filtered teapb2 dimensions: {teapb2.shape}")

## Split the MuData into its three modalities

In [ ]:
adata=teapb2.copy()

In [ ]:
# Extracting RNA, ATAC, and protein data
rna_adata = adata.mod['rna']
atac_adata = adata.mod['atac']
prot_adata = adata.mod['protein']  # Add this line

# Set the obs index
rna_adata.obs.index = adata.obs.index
atac_adata.obs.index = adata.obs.index
prot_adata.obs.index = adata.obs.index  # Add this line

## RNA quality control

In [ ]:
### Top ranking expressed gene ###

sc.pl.highest_expr_genes(rna_adata, n_top=20, )

In [ ]:
### Three plot summary BEFORE filtering ###

rna_adata.var['mt'] = rna_adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(rna_adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

rna_adata.var_names_make_unique()

sc.pl.violin(rna_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

rna_adata

In [ ]:
### Summary of genes and counts BEFORE filtering ###

sc.pl.scatter(rna_adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(rna_adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
# RNA cell/feature filters. These are the thresholds reported in
# Methods; most cells already pass them because the barcode list was
# restricted above.

sc.pp.filter_cells(rna_adata, min_genes=200)
sc.pp.filter_genes(rna_adata, min_cells=3)
rna_adata = rna_adata[rna_adata.obs.n_genes_by_counts < 5000, :]
rna_adata = rna_adata[rna_adata.obs.n_genes_by_counts > 350, :]
rna_adata = rna_adata[rna_adata.obs.pct_counts_mt < 40, :]

In [ ]:
### Three plot summary AFTER filtering ###

sc.pl.violin(rna_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

rna_adata

In [ ]:
### Summary of genes and counts AFTER filtering ###

sc.pl.scatter(rna_adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(rna_adata, x='total_counts', y='n_genes_by_counts')

## Annotate RNA features with their genomic interval

In [ ]:
import pandas as pd

split_interval = rna_adata.var["interval"].str.split(":", expand=True)
rna_adata.var["chr"] = split_interval[0]
split_start_end = split_interval[1].str.split("-", expand=True)

# Convert to numeric, turn errors into NaN
rna_adata.var["start"] = pd.to_numeric(split_start_end[0], errors='coerce').astype('Int64')
rna_adata.var["end"] = pd.to_numeric(split_start_end[1], errors='coerce').astype('Int64')

# Drop rows with NaN values
rna_adata = rna_adata[:, rna_adata.var.dropna().index]

## ATAC feature filtering

In [ ]:
print(atac_adata.shape)
# compute the threshold: 0.5% of the cells
min_cells = int(atac_adata.shape[0] * 0.005)
# in-place filtering of regions
sc.pp.filter_genes(atac_adata, min_cells=min_cells)
print(atac_adata.shape)

In [ ]:
split_interval = atac_adata.var["gene_ids"].str.split(":", expand=True)
atac_adata.var["chr"] = split_interval[0]
split_start_end = split_interval[1].str.split("-", expand=True)
atac_adata.var["start"] = split_start_end[0].astype(int)
atac_adata.var["end"] = split_start_end[1].astype(int)
atac_adata.var

In [ ]:
# Filter out non-chromosomal regions
mask = atac_adata.var["chr"].str.startswith("chr")
atac_adata = atac_adata[:, mask].copy()

## Keep the cells shared by all three modalities

In [ ]:
# Find the common observations
common_obs = rna_adata.obs_names.intersection(atac_adata.obs_names)

# Subset the rna_adata and prot_adata objects to keep only the common observations
rna_adata = rna_adata[common_obs]
atac_adata = atac_adata[common_obs]
prot_adata = prot_adata[common_obs]

## Protein (ADT) feature clean-up

The TotalSeq panel includes isotype-control IgG antibodies (mouse / rat IgG1, IgG2a, IgG2b, and similar). These do not measure a human cell-surface protein; they capture non-specific binding. They are removed before MultiVI so that the protein channel contains only biological ADTs, matching Methods.


In [ ]:
# Split the index on the '-' character and keep only the part before the '-'
prot_adata.var['gene_ids'] = prot_adata.var.index.str.split('-').str[0]
                                                                                                                                                                                                                                                                                                                                                                            
# Add 'feature_types' category and set it to 'Antibody Capture' for all proteins
prot_adata.var['feature_types'] = 'Antibody Capture'

prot_adata.var

In [ ]:
# Drop isotype-control IgG ADTs (non-specific binding, not human surface markers).
prot_adata = prot_adata[:, ~prot_adata.var.index.str.contains('IgG')]
prot_adata.obs


## Assemble the joint RNA + ATAC AnnData for MultiVI

In [ ]:
import scvi
import anndata

# Concatenate RNA and ATAC datasets along the variables axis
adata = anndata.concat([rna_adata, atac_adata], 
                       join='outer', 
                       axis=1, 
                       merge='same')

# Update raw with the raw count data
adata.raw = adata

# Prepare the dataset for scvi-tools
adata.layers["counts"] = adata.X.copy() # preserve counts
adata = adata.raw.to_adata() # revert .raw filtering
adata.layers["counts"] = adata.X.copy() # preserve counts

# Add protein expression data and protein names from prot_adata
adata.obsm['protein_expression'] = prot_adata.X
adata.uns['protein_names'] = prot_adata.var['gene_ids']

In [ ]:
adata.obs["batch_id"] = "1"
# Split the index by "_" and take the second part as the batch_id
adata.obs["batch_id"] = [i.split("_")[1] for i in adata.obs.index]
adata.obs["modality"] = "paired"

In [ ]:
adata_mvi=adata
adata_mvi

In [ ]:
# Copy 'feature_types' to 'modality'
adata_mvi.var['modality'] = adata_mvi.var['feature_types']

In [ ]:
adata_mvi = adata_mvi[:, adata_mvi.var["modality"].argsort()].copy()
adata_mvi.var

## Train MultiVI

In [ ]:
scvi.model.MULTIVI.setup_anndata(
    adata,
    batch_key="modality",  # Use modality as the main batch key
    protein_expression_obsm_key="protein_expression",  # Protein data
    protein_names_uns_key="protein_names"  # Protein feature names
)

In [ ]:
# Create MultiVI model
model = scvi.model.MULTIVI(
    adata,
    n_genes=(adata.var["modality"] == "Gene Expression").sum(),
    n_regions=(adata.var["modality"] == "Peaks").sum(),
)
model.view_anndata_setup()

In [ ]:
model.train()

In [ ]:
import os

# Specify the directory where you want to save the model
save_dir = OUT_DIR / "MultiVI_model"

# One sub-directory per sample so several patients can coexist.
model_dir = save_dir / "MultiVI_PB2"

# Save the model
model.save(model_dir, overwrite=True)

## Optional: skip training and load a saved model

If the model above has already been trained, re-running the notebook only requires loading it back from `model_dir`.

In [ ]:
model = scvi.model.MULTIVI.load(model_dir, adata=adata_mvi)

## Embed and cluster

A nearest-neighbour graph is built on the MultiVI latent space. Leiden community detection is then run at resolution 0.8.

Leiden clustering was initiated at resolution 0.8 with cluster assignments finalized as described in Methods. After that:

1. Inspect global chromatin accessibility (counts per cell) per cluster (next cells). The candidate CSC / root cluster is the one with the highest accessibility.
2. Continue with trajectory / root identification (`02_Trajectory_analysis`) to settle the high-accessibility root / terminal labels used in the figures.

ForceAtlas2 is used only to visualise the latent neighbourhood graph.


In [ ]:
MULTIVI_LATENT_KEY = "X_multivi"

adata_mvi.obsm[MULTIVI_LATENT_KEY] = model.get_latent_representation()
sc.pp.neighbors(adata_mvi, use_rep=MULTIVI_LATENT_KEY)
sc.tl.umap(adata_mvi, min_dist=0.2)
sc.pl.umap(adata_mvi, color="modality")

In [ ]:
# Starting Leiden resolution (Methods). Not the final published cluster labels.
# Inspect peaks/cell below, then refine in notebook 02 (root cluster + trajectory).
sc.tl.leiden(adata_mvi, resolution=0.8, key_added="MULTIVI_LATENT_KEY")


In [ ]:
sc.tl.draw_graph(adata_mvi, layout='fa')

In [ ]:
muon.pl.embedding(
    adata_mvi,
    basis="X_draw_graph_fa",
    color=["MULTIVI_LATENT_KEY"],
    frameon=False,
    ncols=2,
    size=100
)

## Per-cell feature counts and per-cluster accessibility

Use these plots identify the cluster with elevated chromatin accessibility. That cluster is the candidate root / CSC population.


In [ ]:
import numpy as np

# Extract the feature types
feature_types = adata_mvi.var['feature_types']

# Calculate the number of peaks and genes detected per cell using sparse operations
adata_mvi.obs['n_peaks'] = np.array(adata_mvi[:, feature_types == 'Peaks'].X.sum(axis=1)).flatten()
adata_mvi.obs['n_genes'] = np.array(adata_mvi[:, feature_types == 'Gene Expression'].X.sum(axis=1)).flatten()

# Plot the number of peaks detected per cell
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(adata_mvi.obs['n_peaks'], bins=50, color='blue', alpha=0.7, label='Peaks')
plt.hist(adata_mvi.obs['n_genes'], bins=50, color='green', alpha=0.7, label='Genes')
plt.xlabel('Number of Features Detected')
plt.ylabel('Number of Cells')
plt.title('Distribution of Features Detected per Cell')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

adata = adata_mvi.copy()

def plot_cluster_openness(adata, metric):
    # Check if the metric is valid
    if metric not in ['n_peaks', 'n_genes']:
        print(f"Invalid metric: {metric}. Choose either 'n_peaks' or 'n_genes'.")
        return

    # Get the data
    cluster_data = adata.obs.groupby('MULTIVI_LATENT_KEY')[metric].mean()

    # Get the colors from `MULTIVI_LATENT_KEY_colors`
    colors = adata.uns['MULTIVI_LATENT_KEY_colors']
    color_map = {str(i): colors[i] for i in range(len(colors))}

    # Create the plot
    plt.figure(figsize=(10, 5))
    sns.barplot(
        x=cluster_data.index,
        y=cluster_data.values,
        palette=[color_map.get(str(key), 'gray') for key in cluster_data.index]
    )
    if metric == 'n_peaks':
        plt.title('Accessible ATAC peaks per cell by cluster')
        plt.ylabel('Mean accessible ATAC peaks per cell')
    else:
        plt.title(f'Average {metric} per MULTIVI_LATENT_KEY')
        plt.ylabel(f'Average {metric}')
    plt.xlabel('Cluster')
    plt.show()

plot_cluster_openness(adata, 'n_peaks')

## RNA complexity on the ForceAtlas map

Optional QC view: number of genes detected per cell on the same layout. Not used to define clusters or leukemic identity.


In [ ]:
rna_adata.obsm['X_draw_graph_fa']=adata_mvi.obsm['X_draw_graph_fa']

In [ ]:
muon.pl.embedding(
    rna_adata,
    basis="X_draw_graph_fa",
    color=["n_genes_by_counts"],
    frameon=False,
    ncols=1,
    size=20,
)

## Distinguishing leukemic blasts from non-leukemic cells



1. **CellTypist (this notebook)** — transcriptomic annotation with the Developing Human Thymus model, to flag cells whose RNA profile resembles non-leukemic thymic / immune populations.
2. **MTAP / CDKN2A deletion** — leukemic blasts are identified by depletion of scATAC-seq coverage at patient-specific MTAP/CDKN2A intervals, together with loss of MTAP and CDKN2A RNA (Extended Data Fig. 3a,b; Supplementary Table 4; Methods).
3. **Surface-protein profile** — TEA-seq ADTs for clinically annotated markers are compared with the diagnostic flow-cytometry report (Extended Data Fig. 3c; Supplementary Table 1).
4. **T-ALL subtype** — genomic rearrangements plus projection of the pseudobulk transcriptome onto the Polonen et al. AALL0434 reference (Extended Data Fig. 4; Supplementary Table 1).

Together these annotations define the leukemic cells used downstream. This notebook runs step 1. Steps 2–4 are documented in Methods, Extended Data Figs. 3 and 4.


In [ ]:
# Convert 'mt' column to string
adata_mvi.var['mt'] = adata_mvi.var['mt'].astype(str)

# Convert 'protein_names' to a list
adata_mvi.uns['protein_names'] = adata_mvi.uns['protein_names'].tolist()

In [ ]:
import celltypist
from celltypist import models

# CellTypist is the transcriptomic first pass only (see markdown above).
# It runs on the gene-expression modality; majority voting uses the starting Leiden clusters.
adata_celltypist = adata_mvi[
    :, adata_mvi.var["modality"] == "Gene Expression"
].copy()

adata_celltypist.layers["counts"]=adata_celltypist.X.copy()
sc.pp.normalize_total(adata_celltypist, target_sum=1e4)
sc.pp.log1p(adata_celltypist)


In [ ]:
predictions = celltypist.annotate(
    adata_celltypist,
    model='Developing_Human_Thymus.pkl',
    majority_voting=True,
    over_clustering='MULTIVI_LATENT_KEY'
)

In [ ]:
adata_celltypist = predictions.to_adata()

In [ ]:
muon.pl.embedding(
    adata_celltypist,
    basis="X_draw_graph_fa",
    color=["majority_voting"],
    frameon=False,
    ncols=3,
    size=10
)

muon.pl.embedding(
    adata_celltypist,
    basis="X_umap",
    color=["majority_voting"],
    frameon=False,
    ncols=3,
    size=10
)

## Save the cleaned MultiVI object

In [ ]:
# Save the AnnData object
adata_mvi.write(
    DATA_DIR
    / "01_Trimodal_integration_MultiVI/cleaned_MultiVI"
    / "MultiVI_Patient1_Relapse_03h096_Trimodal_Cleaned.h5ad"
)